# Controlled hyperparameter tuning

**Start with the preview. New generation/training is disabled by default.**

This notebook reuses `tuning.py` and the toolkit's existing training/synthesis functions. Original notebooks, configs, and baseline artifacts stay intact. The companion [evaluation notebook](transaction_overdraft_evaluation.ipynb) evaluates saved outputs on CPU.

One parameter changes at a time. Guidance reuses models; classifier changes reuse diffusion; diffusion changes retrain both models; clustering changes rebuild all dependent stages. See [README](README.md) for simple parameter meanings and [PROGRESS](PROGRESS.md) for decisions and findings.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display as show


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/transaction-evaluation-matplotlib")
MULTI_TABLE = ROOT / "implementations/tabular_data/multi_table"

from implementations.tabular_data.multi_table.evaluation.transaction_evaluation import evaluate_saved_runs
from implementations.tabular_data.multi_table.experiments.tuning import (
    PRESETS,
    base_config,
    make_trials,
    preview_trials,
    run_trials,
)

## 1. Choose one experiment

`guidance` defaults to the legacy saved models. Other presets use fresh models. To establish the corrected fresh baseline and sweep guidance on it, set `USE_LEGACY_MODELS = False`.

Fresh models use the same **70/20/10** internal data split for diffusion and classifier. The legacy diffusion used 99/0.5/0.5 while its classifier used 70/20/10. Because each fits preprocessing separately, that mismatch may affect guidance. A fresh model is a **new baseline**, not a controlled extension of the old sweep.

These internal splits do not provide a relational outer holdout. Keep downstream utility claims exploratory until that split is designed before training.

In [ ]:
EXPERIMENT = "guidance"  # See the preset table below.
USE_LEGACY_MODELS = EXPERIMENT == "guidance"
SAMPLING_SEEDS = [42]  # Later: [42, 123, 456] for generation variability.
TRAINING_SEEDS = [42]  # Vary separately for clustering/model variability.
RUN_NEW_TRIALS = False
EXPERIMENT_ROOT = MULTI_TABLE / "results/experiments/transaction_tuning"
DATA_DIR = MULTI_TABLE / "data/berka"

preset_table = pd.DataFrame(
    [
        {"experiment": name, "parameter": parameter, "candidate_values": str(values)}
        for name, (parameter, values) in PRESETS.items()
    ],
)
show(preset_table)

## 2. Preview the cost and reuse

Edit `VALUES` to narrow a sweep. Network-size candidates are lists of layer widths. Defaults are a small starting search, not an optimized recommendation. There is no automatic Cartesian grid across different hyperparameters.

The IDs below identify logical reuse groups. The execution cache additionally fingerprints data, source code, dependency versions, device, and legacy checkpoints. Changing those inputs prevents silent reuse.

In [ ]:
PARAMETER, VALUES = PRESETS[EXPERIMENT]
if USE_LEGACY_MODELS and not PARAMETER.startswith("sampling_config."):
    raise ValueError("Use fresh models for clustering or training changes.")
BASE = base_config(fresh_training=not USE_LEGACY_MODELS)
TRIALS = make_trials(
    BASE,
    PARAMETER,
    VALUES,
    sampling_seeds=SAMPLING_SEEDS,
    training_seeds=TRAINING_SEEDS,
    fresh_training=not USE_LEGACY_MODELS,
)
PLAN = preview_trials(TRIALS)
show(PLAN.drop(columns="config"))
COUNTS = PLAN[["cluster", "diffusion", "model", "sample"]].nunique()
print(f"Planned generations: {COUNTS['sample']}")
if USE_LEGACY_MODELS:
    print("Training needed: none; reuse legacy checkpoints.")
else:
    print(
        f"Before cache reuse: {COUNTS['cluster']} clustering configurations, "
        f"{COUNTS['diffusion']} diffusion sets, {COUNTS['model']} classifier sets.",
    )
    print("Each set covers all nine table/parent relations. Preview is not a GPU-time estimate.")

## 3. Execute only after choosing a GPU budget

Change `RUN_NEW_TRIALS` only when ready to spend GPU time. The saved historical guidance run took about **9 minutes per generation** on its L4 environment; new training time has not been measured.

Stages save `manifest.json`, `status.json`, and checksums. Completed stages are verified and reused. A failed/interrupted stage is not a resumable optimizer checkpoint: inspect its status and select a fresh `EXPERIMENT_ROOT` to retry. Run one notebook process at a time because matching capture temporarily wraps the toolkit's in-process function.

In [ ]:
NEW_RUN_DIRS = []
if RUN_NEW_TRIALS:
    EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
    PLAN.drop(columns="config").to_csv(EXPERIMENT_ROOT / f"plan_{EXPERIMENT}.csv", index=False)
    NEW_RUN_DIRS = run_trials(
        TRIALS,
        DATA_DIR,
        EXPERIMENT_ROOT,
        legacy_results=MULTI_TABLE / "results" if USE_LEGACY_MODELS else None,
        device="cuda",
    )
    show(pd.DataFrame({"saved_run": [str(p) for p in NEW_RUN_DIRS]}))
else:
    print("Preview complete. No clustering, training, or generation was launched.")

## 4. Evaluate completed runs with the same framework

The same overdraft detector and reference split are shared across all runs in this call. Compare AP/prevalence/uncertainty, transaction marginals, rare-event conditionals, and account chronology separately. Retain the baseline when evaluating a narrowed candidate set.

For saved historical runs, use the evaluation notebook now. For new runs, the cell below works directly from the returned paths.

In [ ]:
if NEW_RUN_DIRS:
    comparison, report_dir = evaluate_saved_runs(
        DATA_DIR,
        NEW_RUN_DIRS,
        MULTI_TABLE / "results/experiments/transaction_overdraft",
        bootstrap_repetitions=200,
    )
    show(
        comparison[
            [
                "average_precision",
                "standardized_ap",
                "overdraft_prevalence",
                "transaction_ks",
                "transaction_tvd",
                "overdraft_numerical_ks",
                "synthetic_transaction_before_account_rate",
            ]
        ].round(4),
    )
    print(f"Detailed report: {report_dir}")

## Research log

Record the hypothesis, exact changed parameter, reference baseline, generation/training seeds, observed tradeoffs, and next decision in [PROGRESS.md](PROGRESS.md). A good candidate must preserve overdraft patterns and account relationships as well as overall distributions. No universal weighted score or automatic winner is used.

**Validation boundary:** saved-data evaluation has run on CPU; experiment orchestration is tested with lightweight substitutes. Full fresh ClavaDDPM training has not been run under the current GPU budget.